In [6]:
"""Regression 2.2 - optimisation is clarified WITHOUT the change of 8 locked predictors.
Diagnostics found: (1) negative predictions, (2) skewed/non-normal residuals,
(3) mild heteroscedasticity, (4) count target. Each fix below targets one of these."""
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CSV_PATH = "regression_2_2_dataset.csv"
OUT = ""
TARGET, GROUP, SEED, N_REPEATS = "goals_scored", "match_id", 42, 200
FEATURES = ["avg_goals_scored_before", "opponent_avg_goals_conceded_before", "team_fifa_points",
            "opponent_fifa_points", "win_rate_before", "opponent_scoring_match_rate_before",
            "avg_goal_difference_before", "opponent_clean_sheet_rate_before"]
df = pd.read_csv(CSV_PATH); X, y, g = df[FEATURES], df[TARGET], df[GROUP]
nrmse = lambda t, p: np.sqrt(mean_squared_error(t, p)) / (t.max() - t.min())


class Variant:
    """name, estimator, optional target transform (forward, inverse), clip negatives."""
    def __init__(s, name, est, fwd=None, inv=None, clip=False):
        s.name, s.est, s.fwd, s.inv, s.clip = name, est, fwd, inv, clip

    def run(s, Xtr, ytr, Xte):
        m = clone(s.est).fit(Xtr, ytr if s.fwd is None else s.fwd(ytr))
        p = m.predict(Xte)
        p = p if s.inv is None else s.inv(p)
        return np.clip(p, 0, None) if s.clip else p


ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
variants = [
    Variant("Baseline OLS", LinearRegression()),
    Variant("OLS + clip negatives at 0", LinearRegression(), clip=True),
    Variant("OLS on sqrt(goals)", LinearRegression(), np.sqrt, np.square),
    Variant("OLS on log1p(goals)", LinearRegression(), np.log1p, np.expm1),
    Variant("Ridge (CV alpha, scaled)", ridge, clip=True),
]
rows = []
for seed in range(N_REPEATS):
    for ts in (0.2, 0.3):
        tr, te = next(GroupShuffleSplit(1, test_size=ts, random_state=seed).split(df, y, g))
        for v in variants:
            p = v.run(X.iloc[tr], y.iloc[tr], X.iloc[te])
            rows.append(dict(model=v.name, split=f"{round((1-ts)*100)}/{round(ts*100)}", seed=seed,
                             R2_test=r2_score(y.iloc[te], p), NRMSE_test=nrmse(y.iloc[te], p)))
rep = pd.DataFrame(rows)
summ = rep.groupby(["model", "split"])[["R2_test", "NRMSE_test"]].agg(["mean", "std"]).round(4)
print(f"=== OPTIMISATION: {N_REPEATS} repeated grouped splits ===\n", summ.to_string())

# paired comparison vs baseline: how often does each variant beat baseline on the same split?
base = rep[rep.model == "Baseline OLS"].set_index(["split", "seed"])["R2_test"]
print("\n% of splits where variant beats baseline (test R2), and mean paired difference:")
for v in variants[1:]:
    d = rep[rep.model == v.name].set_index(["split", "seed"])["R2_test"] - base
    print(f"  {v.name:30s} {100*(d>0).mean():5.1f}%   mean diff {d.mean():+.4f}")

# single seed-42 table (same type as the main results)
single = rep[rep.seed == SEED].drop(columns="seed").round(4)
print("\n=== Seed 42 ===\n", single.to_string(index=False))
r = RidgeCV(alphas=np.logspace(-2, 3, 30)).fit(StandardScaler().fit_transform(X), y)
print("\nRidge chosen alpha (full data):", round(r.alpha_, 3))
summ.to_csv(OUT + "results_optimisation_repeated.csv"); single.to_csv(OUT + "results_optimisation_seed42.csv", index=False)

=== OPTIMISATION: 200 repeated grouped splits ===
                                 R2_test         NRMSE_test        
                                   mean     std       mean     std
model                     split                                   
Baseline OLS              70/30  0.1705  0.1167     0.2124  0.0242
                          80/20  0.1732  0.1535     0.2262  0.0332
OLS + clip negatives at 0 70/30  0.1744  0.1155     0.2119  0.0242
                          80/20  0.1768  0.1524     0.2258  0.0330
OLS on log1p(goals)       70/30  0.1610  0.1031     0.2139  0.0241
                          80/20  0.1593  0.1302     0.2283  0.0300
OLS on sqrt(goals)        70/30  0.1306  0.1158     0.2176  0.0250
                          80/20  0.1260  0.1489     0.2326  0.0316
Ridge (CV alpha, scaled)  70/30  0.1875  0.0886     0.2105  0.0231
                          80/20  0.1875  0.1229     0.2246  0.0304

% of splits where variant beats baseline (test R2), and mean paired differenc